# Notebook 01 - Data Preprocessing 
**Goal:** load the raw Kaggle BBC News dataset, clean it, and save the preprocessed files.

**Input:** `data/raw/bbc-text.csv` (download from Kaggle, see GUIDE.md)
**Output:** `data/processed/bbc_clean.csv`, `train.csv`, `test.csv`

In [28]:
# Import the libraries we need
import sys  # sys lets us change where Python looks for our own code
from pathlib import Path  # Path builds file paths safely

import pandas as pd  # pandas works with tables (CSV files)
from sklearn.model_selection import train_test_split  # splits data into train and test

sys.path.append("..")  # allow importing our project code from the parent folder
from app.services.preprocessing import clean_for_model  # the SAME cleaning function the API uses

In [29]:
# Set the file locations
RAW_PATH = Path("../data/raw/bbc-text.csv")  # raw file downloaded from Kaggle
OUT_DIR = Path("../data/processed")  # folder where we save the results
OUT_DIR.mkdir(parents=True, exist_ok=True)  # create the folder if it does not exist

## Step 1 - Load and explore the raw data

In [30]:
# Read the raw CSV file into a table
df = pd.read_csv(RAW_PATH)  # columns: category, text
df.head()  # show the first 5 rows

,category,text
0,tech,tv future in the hands of viewers with home th...
1,business,worldcom boss left books alone former worldc...
2,sport,tigers wary of farrell gamble leicester say ...
3,sport,yeading face newcastle in fa cup premiership s...
4,entertainment,ocean s twelve raids box office ocean s twelve...


In [31]:
# Look at the basic facts about the data
print("Shape (rows, columns):", df.shape)  # how big is the data
print("Missing values:\n", df.isnull().sum())  # count empty cells per column
print("Duplicate rows:", df.duplicated().sum())  # count repeated rows
print("\nClass counts:\n", df["category"].value_counts())  # how many articles per category

Shape (rows, columns): (2225, 2)
Missing values:
 category    0
text        0
dtype: int64
Duplicate rows: 99

Class counts:
 category
sport            511
business         510
politics         417
tech             401
entertainment    386
Name: count, dtype: int64


## Step 2 - Remove bad rows and rename the label column

In [32]:
# Drop empty rows and duplicates, then reset the row numbers
df = df.dropna().drop_duplicates().reset_index(drop=True)  # clean the table structure
df = df.rename(columns={"category": "label"})  # a clearer name for the class column
print("Rows after removing bad rows:", len(df))  # check the new size

Rows after removing bad rows: 2126


## Step 3 - Clean the text (lowercase, remove links/numbers/symbols, remove stopwords)

In [33]:
# Create a new column with the cleaned text
df["clean_text"] = df["text"].apply(clean_for_model)  # apply our cleaning function to every row
print("BEFORE:", df.loc[0, "text"][:200])  # show an original article start
print("AFTER :", df.loc[0, "clean_text"][:200])  # show the cleaned version

BEFORE: tv future in the hands of viewers with home theatre systems  plasma high-definition tvs  and digital video recorders moving into the living room  the way people watch tv will be radically different in
AFTER : tv future hands viewers home theatre systems plasma high definition tvs digital video recorders moving into living room way people watch tv radically different five years time according expert panel g


In [34]:
# Remove rows that became empty after cleaning
df = df[df["clean_text"].str.len() > 0].reset_index(drop=True)  # keep only non-empty rows
df["word_count"] = df["clean_text"].str.split().str.len()  # count words per article
print(df["word_count"].describe())  # summary of article lengths

count    2126.000000
mean      248.023989
std       155.278609
min        55.000000
25%       157.000000
50%       214.000000
75%       303.000000
max      2744.000000
Name: word_count, dtype: float64


## Step 4 - Save the preprocessed data

In [35]:
# Save the full preprocessed dataset (row order matters for BERT vectors later!)
df[["text", "clean_text", "label"]].to_csv(OUT_DIR / "bbc_clean.csv", index=False)  # save without row numbers
print("Saved bbc_clean.csv with", len(df), "rows")  # confirm

Saved bbc_clean.csv with 2126 rows


In [36]:
# Split into 80% training and 20% testing (same split is used in every notebook)
train_df, test_df = train_test_split(  # do the split
    df[["text", "clean_text", "label"]], test_size=0.2, random_state=42, stratify=df["label"]  # keep class balance
)
train_df.to_csv(OUT_DIR / "train.csv", index=False)  # save training data
test_df.to_csv(OUT_DIR / "test.csv", index=False)  # save test data
print("Train rows:", len(train_df), "| Test rows:", len(test_df))  # confirm

Train rows: 1700 | Test rows: 426
